# Phase 3: Skill & Entity Extraction

This notebook implements and evaluates the skill and entity extraction pipeline for resumes and job descriptions.

## 1. Environment Setup

In [ ]:
!pip install datasets spacy pandas scikit-learn
!python -m spacy download en_core_web_sm

## 2. Dataset Loading
We load `yashpwr/resume-ner-training-data` to inspect its viability for NER training. We'll also use sample resume and job texts for testing our baseline pipeline.

In [ ]:
import pandas as pd
from datasets import load_dataset

try:
    ner_dataset = load_dataset("yashpwr/resume-ner-training-data", split="train[:100]")
    print("Loaded NER dataset sample.")
except Exception as e:
    print(f"Could not load dataset: {e}")

## 3. Data Inspection
Inspect the schema of the NER dataset.

In [ ]:
if 'ner_dataset' in locals():
    print(ner_dataset[0])
    # It's observed that this dataset often uses conversational JSON strings.
    # Converting these to strict BIO tokens for NER training requires complex parsing
    # and is prone to label misalignment. We will document this and use a robust baseline instead.

## 4. Skill/Entity Definition
We aim to extract:
- **Technical Skills** (Languages, Frameworks, DBs, Cloud)
- **Certifications**
- **Education (Degrees)**
- **Job Roles**
- **Years of Experience**

## 5. Baseline Extraction Approach (Modules)
We rely on our custom modules built in `src/skill_extraction/`: `skill_dictionary`, `normalization`, `rule_based`, and `ner`.

In [ ]:
import sys
import os
# Ensure src is in path
sys.path.append(os.path.abspath('../'))
try:
    from src.skill_extraction.extractor import extract_from_resume, extract_from_job
    print("Loaded extraction modules successfully.")
except ImportError:
    print("Run this locally with src/ available, or copy the modules into Colab cells.")

## 9. Resume Extraction Examples

In [ ]:
sample_resume = """
Experienced Data Scientist with 5+ years of experience.
Proficient in Python, R, and Machine Learning. 
Built backend services using NodeJS and MongoDB.
Hold a Master of Science degree in Computer Science.
"""

if 'extract_from_resume' in locals():
    import json
    res = extract_from_resume(sample_resume, record_id="RES-101")
    print(json.dumps(res, indent=2))

## 10. Job-description Extraction Examples

In [ ]:
sample_job = """
Looking for a Backend Developer.
Requirements:
- 3 years of experience in software engineering.
- Strong knowledge of Node.js, Express.js, and PostgreSQL.
- B.S. in Computer Science or related field.
"""

if 'extract_from_job' in locals():
    import json
    job_res = extract_from_job(sample_job, job_id="JOB-202")
    print(json.dumps(job_res, indent=2))

## 11. Extraction Evaluation
Since standard labelled BIO test sets for our exact schema aren't readily available without manual annotation of `yashpwr` data, we perform a qualitative evaluation and error analysis.

## 12. Error Analysis
**False Positives**: The phrase matcher might pick up common words if they overlap with a skill (e.g., 'Go' as a verb vs 'Go' the programming language, though case sensitivity helps). 
**Missed Skills**: Skills not explicitly in our `skill_dictionary` will be missed unless caught by an advanced NER model.
**Incorrect Normalization**: Ambiguous aliases. For instance, 'JS' might stand for something else in a non-tech context, but we normalize it to 'JavaScript'.

## 13. Structured Output
The output dictionaries follow the exact requested format for integration with the Matching system later.

## 14. Phase 3 Conclusions
1. **NER Dataset**: The `resume-ner-training-data` uses conversational/JSON strings that make standard BIO-tag extraction noisy. A rule-based baseline backed by a rich dictionary is more reliable for a V1.
2. **Normalization**: Canonical mapping (e.g., nodejs -> Node.js) effectively standardizes representations.
3. **Integration**: The pipeline exposes `extract_from_resume` and `extract_from_job` taking simple string inputs, ensuring compatibility with Phase 2's `cleaned_text` output.